# 14 — Structured Factor Stack V1

Independent replacement candidate for Production Control.

- Daewoon and Sewoon are learned separately.
- 11 semantic Stage-1 factor heads.
- Stage 2 combines only cross-fitted factor scores.
- Exactly one Daewoon × Sewoon interaction.
- Control, axis, age, chronology and nuisance are forbidden predictors.
- 101 pairs are consumed DEV only. If the frozen DEV sanity gate fails, reject this architecture and do not collect fresh confirmation data.

In [1]:
from pathlib import Path
from datetime import datetime
from collections import OrderedDict
import hashlib, json, pickle, warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
warnings.filterwarnings('ignore')

NOTEBOOK_VERSION='SAJU_ML_V4_STRUCTURED_FACTOR_STACK_V1_20260817'
SEED=20260816; OUTER_FOLDS=5; OUTER_REPEATS=10; INNER_FOLDS=3; N_BOOTSTRAP=20000
C_STAGE1=0.3; C_STAGE2=0.3
MIN_OVERALL=0.55; MIN_DELTA_CONTROL=0.0; MIN_P10=0.48; MIN_AXIS=0.45

def find_repo_root(start=None):
    p=Path(start or Path.cwd()).resolve()
    for c in [p]+list(p.parents):
        if (c/'saju_engine.py').exists(): return c
    raise FileNotFoundError('Run inside the Chartpalja repository.')

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

ROOT=find_repo_root()
SRC=ROOT/'research/ml/artifacts/v4_unified_dev_3arch_tournament'
DIAG=ROOT/'research/ml/artifacts/v4_control_failure_diagnostics'
SPEC_PATH=ROOT/'research/ml_corpus/v4_structured_factor_stack/V4_STRUCTURED_FACTOR_STACK_V1_SPEC.json'
PAIR_PATH=SRC/'V4_UNIFIED_DEV_pair_diff_feature_table.csv'
YEAR_PATH=SRC/'V4_UNIFIED_DEV_year_features.csv'
TOURNAMENT_DECISION=SRC/'V4_UNIFIED_DEV_3ARCH_TOURNAMENT_DECISION.json'
FEATURE_MANIFEST=SRC/'V4_UNIFIED_DEV_FEATURE_GENERATION_MANIFEST.json'
DIAG_DECISION=DIAG/'V4_CONTROL_FAILURE_DIAGNOSTIC_DECISION.json'
ARTIFACT_DIR=ROOT/'research/ml/artifacts/v4_structured_factor_stack_v1'; ARTIFACT_DIR.mkdir(parents=True,exist_ok=True)
for p in [SPEC_PATH,PAIR_PATH,YEAR_PATH,TOURNAMENT_DECISION,FEATURE_MANIFEST,DIAG_DECISION]:
    if not p.exists(): raise FileNotFoundError(p)
spec=json.load(open(SPEC_PATH,encoding='utf-8'))
old=json.load(open(TOURNAMENT_DECISION,encoding='utf-8'))
diag=json.load(open(DIAG_DECISION,encoding='utf-8'))
pair_df=pd.read_csv(PAIR_PATH); year_df=pd.read_csv(YEAR_PATH)
assert spec['status']=='PREREGISTERED_BEFORE_RUN'
assert old['status']=='V4_UNIFIED_DEV_NO_ARCHITECTURE_SURVIVES_DISCOVERY_GATE'
assert diag['status']=='V4_CONTROL_FAILURE_DIAGNOSTICS_COMPLETE_REDESIGN_BEFORE_FRESH_CONFIRM_DEV'
assert len(pair_df)==101 and pair_df.subject_id.nunique()==101
assert len(year_df)==202 and year_df[['subject_id','year']].drop_duplicates().shape[0]==202
assert all(v is False for v in old['holdout_integrity'].values())
assert all(v is False for v in diag['holdout_integrity'].values())
CONTROL_DIFF='diff__baseline__control_score'; assert CONTROL_DIFF in pair_df.columns
print('Preflight PASS', ROOT)

Preflight PASS /Users/sangjinlee/Desktop/projects/saju


## 1. Frozen Stage-1 semantic blocks

In [2]:
year_feature_cols=[c for c in year_df.columns if c not in {'subject_id','year'}]
def nonzero(cols):
    out=[]
    for c in cols:
        d='diff__'+c
        if d in pair_df.columns and pd.to_numeric(pair_df[d],errors='coerce').fillna(0).var()>0: out.append(c)
    return sorted(out)
def block(prefix,marker): return nonzero([c for c in year_feature_cols if c.startswith(prefix) and marker in c])
BLOCKS=OrderedDict([
 ('DW_BASIC',block('basic__','basic__dw_')),('DW_TG10',block('tg10__','tg10__dw_')),('DW_YONGSHIN',block('yongshin__','yongshin__dw_')),('DW_UNSEONG',block('unseong__','unseong__dw_')),('DW_RELATION',block('relation__','relation__dw_')),
 ('SW_BASIC',block('basic__','basic__sw_')),('SW_TG10',block('tg10__','tg10__sw_')),('SW_YONGSHIN',block('yongshin__','yongshin__sw_')),('SW_UNSEONG',block('unseong__','unseong__sw_')),('SW_RELATION',block('relation__','relation__sw_')),
 ('CTX_DW_SW_RELATION',nonzero([c for c in year_feature_cols if c.startswith('relation__sw_dw_')]))])
BLOCKS['SW_RELATION']=[c for c in BLOCKS['SW_RELATION'] if not c.startswith('relation__sw_dw_')]
for k,v in BLOCKS.items():
    if not v: raise RuntimeError('Frozen block %s has zero varying features'%k)
    bad=[c for c in v if any(t in c.lower() for t in ['control','age','axis','chronology','nuisance','event','label','origin'])]
    assert not bad,(k,bad)
json.dump(BLOCKS,open(ARTIFACT_DIR/'V4_STRUCTURED_FACTOR_STACK_V1_BLOCK_SCHEMA.json','w',encoding='utf-8'),ensure_ascii=False,indent=2)
display(pd.DataFrame([{'block':k,'n_features':len(v)} for k,v in BLOCKS.items()]))

,block,n_features
0,DW_BASIC,29
1,DW_TG10,20
2,DW_YONGSHIN,8
3,DW_UNSEONG,14
4,DW_RELATION,7
5,SW_BASIC,29
6,SW_TG10,20
7,SW_YONGSHIN,8
8,SW_UNSEONG,14
9,SW_RELATION,7


## 2. Pairwise factor learners and annual latent factors

In [3]:
def fit_pairwise(frame,year_cols,C):
    X=frame[['diff__'+c for c in year_cols]].to_numpy(float)
    X=np.vstack([X,-X]); y=np.r_[np.ones(len(frame),int),np.zeros(len(frame),int)]
    m=Pipeline([('scale',StandardScaler()),('model',LogisticRegression(penalty='l2',C=C,solver='liblinear',random_state=SEED,max_iter=5000))])
    m.fit(X,y); return m

def fit_meta(frame,cols,C):
    X=frame[cols].to_numpy(float); X=np.vstack([X,-X]); y=np.r_[np.ones(len(frame),int),np.zeros(len(frame),int)]
    m=Pipeline([('scale',StandardScaler()),('model',LogisticRegression(penalty='l2',C=C,solver='liblinear',random_state=SEED,max_iter=5000))])
    m.fit(X,y); return m

def fit_stage1(frame): return OrderedDict((k,fit_pairwise(frame,v,C_STAGE1)) for k,v in BLOCKS.items())
year_lookup=year_df.set_index(['subject_id','year'])

def annual_factor_vector(models,sid,year):
    row=year_lookup.loc[(sid,int(year))]; out={}
    for k,m in models.items():
        out[k]=float(m.decision_function(row[BLOCKS[k]].to_numpy(float).reshape(1,-1))[0])
    dw=[out[k] for k in out if k.startswith('DW_')]; sw=[out[k] for k in out if k.startswith('SW_')]
    out['META_DW_SW_INTERACTION']=float(np.mean(dw)*np.mean(sw))
    return out
META_COLS=list(BLOCKS.keys())+['META_DW_SW_INTERACTION']

def build_meta_pairs(frame,models):
    rows=[]
    for _,r in frame.iterrows():
        p=annual_factor_vector(models,r.subject_id,r.positive_year); n=annual_factor_vector(models,r.subject_id,r.negative_year)
        z={'pair_id':r.pair_id,'subject_id':r.subject_id,'axis':r.axis,'positive_earlier_calc':bool(r.positive_earlier_calc)}
        for c in META_COLS: z[c]=float(p[c]-n[c])
        rows.append(z)
    return pd.DataFrame(rows)

## 3. Leakage-safe nested cross-fitting

In [4]:
def splits(frame,n_folds,seed):
    m=frame[['subject_id','axis','positive_earlier_calc']].copy(); assert m.subject_id.nunique()==len(m)
    m['stratum']=m.axis.astype(str)+'__'+m.positive_earlier_calc.astype(int).astype(str)
    cnt=m.stratum.value_counts()
    if cnt.min()<n_folds: raise RuntimeError('Cannot stratify: '+str(cnt.to_dict()))
    sk=StratifiedKFold(n_splits=n_folds,shuffle=True,random_state=seed); X=np.zeros((len(m),1)); y=m.stratum.to_numpy(); out=[]
    for tr,te in sk.split(X,y): out.append((m.iloc[tr].subject_id.tolist(),m.iloc[te].subject_id.tolist()))
    return out

def crossfit_meta_train(train,seed):
    tmp=train.copy(); tmp['_s']=tmp.axis.astype(str)+'__'+tmp.positive_earlier_calc.astype(int).astype(str)
    n_inner=min(INNER_FOLDS,int(tmp.groupby('_s').size().min()))
    if n_inner<2: raise RuntimeError('Insufficient inner strata')
    pieces=[]
    for fi,(fit_ids,val_ids) in enumerate(splits(train,n_inner,seed)):
        fit=train[train.subject_id.isin(fit_ids)]; val=train[train.subject_id.isin(val_ids)]
        meta=build_meta_pairs(val,fit_stage1(fit)); meta['inner_fold']=fi; pieces.append(meta)
    out=pd.concat(pieces,ignore_index=True); assert len(out)==len(train) and out.subject_id.nunique()==len(train); return out

outer_rows=[]
for rep in range(OUTER_REPEATS):
    for fold,(train_ids,test_ids) in enumerate(splits(pair_df,OUTER_FOLDS,SEED+1000*rep)):
        train=pair_df[pair_df.subject_id.isin(train_ids)].copy(); test=pair_df[pair_df.subject_id.isin(test_ids)].copy()
        meta_train=crossfit_meta_train(train,SEED+10000*rep+fold); stage2=fit_meta(meta_train,META_COLS,C_STAGE2)
        stage1=fit_stage1(train); meta_test=build_meta_pairs(test,stage1)
        margin=stage2.decision_function(meta_test[META_COLS].to_numpy(float)); cand=(margin>0).astype(float)
        cm=test[CONTROL_DIFF].to_numpy(float); ctrl=(cm>0).astype(float)
        for i,(_,r) in enumerate(test.iterrows()):
            outer_rows.append({'repeat':rep,'fold':fold,'pair_id':r.pair_id,'subject_id':r.subject_id,'axis':r.axis,'candidate_margin':float(margin[i]),'candidate_correct':float(cand[i]),'control_margin':float(cm[i]),'control_correct':float(ctrl[i])})
    print('repeat',rep+1,'/',OUTER_REPEATS)
outer=pd.DataFrame(outer_rows); outer.to_csv(ARTIFACT_DIR/'V4_STRUCTURED_FACTOR_STACK_V1_OUTER_CV.csv',index=False)
assert outer.groupby(['repeat','subject_id']).size().eq(1).all()

repeat 1 / 10
repeat 2 / 10
repeat 3 / 10
repeat 4 / 10
repeat 5 / 10
repeat 6 / 10
repeat 7 / 10
repeat 8 / 10
repeat 9 / 10
repeat 10 / 10


## 4. Development sanity gate

In [5]:
rep=(outer.groupby('repeat').agg(candidate=('candidate_correct','mean'),control=('control_correct','mean')).reset_index()); rep['delta']=rep.candidate-rep.control
rep.to_csv(ARTIFACT_DIR/'V4_STRUCTURED_FACTOR_STACK_V1_REPEAT_SUMMARY.csv',index=False); display(rep)
candidate_mean=float(rep.candidate.mean()); control_mean=float(rep.control.mean()); delta=candidate_mean-control_mean; p10=float(np.quantile(rep.candidate,.10))
axis=(outer.groupby(['repeat','axis']).agg(candidate=('candidate_correct','mean'),control=('control_correct','mean')).reset_index().groupby('axis').agg(candidate=('candidate','mean'),control=('control','mean')).reset_index()); axis['delta']=axis.candidate-axis.control
axis.to_csv(ARTIFACT_DIR/'V4_STRUCTURED_FACTOR_STACK_V1_AXIS_DIAGNOSTICS.csv',index=False); display(axis)
axis_floor=float(axis.candidate.min())
subject_avg=outer.groupby('subject_id').agg(candidate=('candidate_correct','mean'),control=('control_correct','mean'))
rng=np.random.RandomState(SEED+777); ids=subject_avg.index.to_numpy(); bd=[]
for _ in range(N_BOOTSTRAP):
    s=rng.choice(ids,size=len(ids),replace=True); bd.append(float((subject_avg.loc[s,'candidate'].to_numpy()-subject_avg.loc[s,'control'].to_numpy()).mean()))
bd=np.asarray(bd)
boot=pd.DataFrame([{'observed_delta_vs_control':float((subject_avg.candidate-subject_avg.control).mean()),'ci025':float(np.quantile(bd,.025)),'ci975':float(np.quantile(bd,.975)),'p_delta_gt_0':float((bd>0).mean())}]); boot.to_csv(ARTIFACT_DIR/'V4_STRUCTURED_FACTOR_STACK_V1_BOOTSTRAP.csv',index=False); display(boot)
gate={'overall_ge_055':candidate_mean>=MIN_OVERALL,'delta_vs_control_ge_000':delta>=MIN_DELTA_CONTROL,'p10_ge_048':p10>=MIN_P10,'axis_floor_ge_045':axis_floor>=MIN_AXIS}; dev_pass=all(gate.values())
print({'candidate':candidate_mean,'control':control_mean,'delta':delta,'p10':p10,'axis_floor':axis_floor,'gate':gate,'DEV_PASS':dev_pass})

,repeat,candidate,control,delta
0,0,0.534653,0.524752,0.009901
1,1,0.584158,0.524752,0.059406
2,2,0.465347,0.524752,-0.059406
3,3,0.485149,0.524752,-0.039604
4,4,0.495050,0.524752,-0.029703
5,5,0.544554,0.524752,0.019802
6,6,0.495050,0.524752,-0.029703
7,7,0.435644,0.524752,-0.089109
8,8,0.455446,0.524752,-0.069307
9,9,0.524752,0.524752,0.000000


,axis,candidate,control,delta
0,COMPETITIVE,0.496774,0.516129,-0.019355
1,PROJECT,0.480000,0.600000,-0.120000
2,STATUS,0.517778,0.488889,0.028889


,observed_delta_vs_control,ci025,ci975,p_delta_gt_0
0,-0.022772,-0.134653,0.088119,0.34455


{'candidate': 0.501980198019802, 'control': 0.5247524752475247, 'delta': -0.02277227722772268, 'p10': 0.4534653465346535, 'axis_floor': 0.48, 'gate': {'overall_ge_055': False, 'delta_vs_control_ge_000': False, 'p10_ge_048': False, 'axis_floor_ge_045': True}, 'DEV_PASS': False}


## 5. Conditional freeze on all consumed DEV

In [6]:
def export_pipe(m,cols):
    s=m.named_steps['scale']; lr=m.named_steps['model']
    return {'columns':list(cols),'scaler_mean':[float(x) for x in s.mean_],'scaler_scale':[float(x) for x in s.scale_],'coefficient':[float(x) for x in lr.coef_.ravel()],'intercept':float(lr.intercept_[0]),'C':float(lr.C)}
model_path=ARTIFACT_DIR/'V4_STRUCTURED_FACTOR_STACK_V1_FROZEN_MODELS.pkl'; weights_path=ARTIFACT_DIR/'V4_STRUCTURED_FACTOR_STACK_V1_FROZEN_WEIGHTS.json'; coef_path=ARTIFACT_DIR/'V4_STRUCTURED_FACTOR_STACK_V1_STAGE2_COEFFICIENTS.csv'
if dev_pass:
    pieces=[]
    for fold,(fit_ids,val_ids) in enumerate(splits(pair_df,OUTER_FOLDS,SEED+4242)):
        fit=pair_df[pair_df.subject_id.isin(fit_ids)]; val=pair_df[pair_df.subject_id.isin(val_ids)]; m=build_meta_pairs(val,fit_stage1(fit)); m['fold']=fold; pieces.append(m)
    full_meta=pd.concat(pieces,ignore_index=True); assert len(full_meta)==101 and full_meta.subject_id.nunique()==101
    s2=fit_meta(full_meta,META_COLS,C_STAGE2); s1=fit_stage1(pair_df)
    pickle.dump({'architecture':'STRUCTURED_FACTOR_STACK_V1','stage1':s1,'stage2':s2,'blocks':BLOCKS,'meta_cols':META_COLS},open(model_path,'wb'))
    transparent={'architecture':'STRUCTURED_FACTOR_STACK_V1','stage1':{k:export_pipe(s1[k],BLOCKS[k]) for k in BLOCKS},'stage2':export_pipe(s2,META_COLS),'annual_score_policy':'Stage-2 decision_function is latent annual score; no 0-100 calibration before fresh confirmation.'}
    json.dump(transparent,open(weights_path,'w',encoding='utf-8'),ensure_ascii=False,indent=2)
    pd.DataFrame({'meta_factor':META_COLS,'coefficient':s2.named_steps['model'].coef_.ravel()}).assign(abscoef=lambda x:x.coefficient.abs()).sort_values('abscoef',ascending=False).drop(columns='abscoef').to_csv(coef_path,index=False)
    status='V4_STRUCTURED_FACTOR_STACK_V1_FROZEN_READY_FOR_FRESH_CONFIRM_DEV'
else: status='V4_STRUCTURED_FACTOR_STACK_V1_REJECTED_NO_FRESH_CONFIRM'
print(status)

V4_STRUCTURED_FACTOR_STACK_V1_REJECTED_NO_FRESH_CONFIRM


## 6. Decision and lineage

In [7]:
lineage={'created_at':datetime.now().isoformat(timespec='seconds'),'notebook_version':NOTEBOOK_VERSION,'spec_sha256':sha256_file(SPEC_PATH),'pair_diff_sha256':sha256_file(PAIR_PATH),'year_features_sha256':sha256_file(YEAR_PATH),'saju_engine_sha256':sha256_file(ROOT/'saju_engine.py'),'frozen_model_sha256':sha256_file(model_path) if model_path.exists() else None,'frozen_weights_sha256':sha256_file(weights_path) if weights_path.exists() else None}
json.dump(lineage,open(ARTIFACT_DIR/'V4_STRUCTURED_FACTOR_STACK_V1_LINEAGE.json','w',encoding='utf-8'),ensure_ascii=False,indent=2)
decision={'version':'V4_STRUCTURED_FACTOR_STACK_V1_DECISION','status':status,'architecture':'STRUCTURED_FACTOR_STACK_V1','corpus_role':'CONSUMED_ARCHITECTURE_DEVELOPMENT','n_pairs':101,'metrics':{'candidate_outer_cv_mean':candidate_mean,'control_fixed_mean':control_mean,'delta_vs_control':delta,'candidate_outer_cv_p10':p10,'candidate_axis_floor':axis_floor,'bootstrap_P_delta_gt_0_vs_control':float(boot.iloc[0].p_delta_gt_0)},'gate':gate,'dev_sanity_pass':bool(dev_pass),'candidate_predictor_exclusions':['Control','axis','age','chronology','nuisance','event metadata'],'holdout_integrity':{'NEW_CONFIRM_loaded':False,'Validation_B_loaded':False,'Public_CHECK_loaded':False,'Public_FINAL_loaded':False},'next_rule':'If frozen, do not change architecture or coefficients; build completely fresh confirmation DEV and compare frozen candidate vs Production Control once. If rejected, do not collect fresh confirmation for this architecture.'}
json.dump(decision,open(ARTIFACT_DIR/'V4_STRUCTURED_FACTOR_STACK_V1_DECISION.json','w',encoding='utf-8'),ensure_ascii=False,indent=2)
print(json.dumps(decision,ensure_ascii=False,indent=2))

{
  "version": "V4_STRUCTURED_FACTOR_STACK_V1_DECISION",
  "status": "V4_STRUCTURED_FACTOR_STACK_V1_REJECTED_NO_FRESH_CONFIRM",
  "architecture": "STRUCTURED_FACTOR_STACK_V1",
  "corpus_role": "CONSUMED_ARCHITECTURE_DEVELOPMENT",
  "n_pairs": 101,
  "metrics": {
    "candidate_outer_cv_mean": 0.501980198019802,
    "control_fixed_mean": 0.5247524752475247,
    "delta_vs_control": -0.02277227722772268,
    "candidate_outer_cv_p10": 0.4534653465346535,
    "candidate_axis_floor": 0.48,
    "bootstrap_P_delta_gt_0_vs_control": 0.34455
  },
  "gate": {
    "overall_ge_055": false,
    "delta_vs_control_ge_000": false,
    "p10_ge_048": false,
    "axis_floor_ge_045": true
  },
  "dev_sanity_pass": false,
  "candidate_predictor_exclusions": [
    "Control",
    "axis",
    "age",
    "chronology",
    "nuisance",
    "event metadata"
  ],
  "holdout_integrity": {
    "NEW_CONFIRM_loaded": false,
    "Validation_B_loaded": false,
    "Public_CHECK_loaded": false,
    "Public_FINAL_loaded": f

## Send back

After **Kernel Restart → Run All**, send exactly:

```text
V4_STRUCTURED_FACTOR_STACK_V1_DECISION.json
V4_STRUCTURED_FACTOR_STACK_V1_REPEAT_SUMMARY.csv
V4_STRUCTURED_FACTOR_STACK_V1_AXIS_DIAGNOSTICS.csv
V4_STRUCTURED_FACTOR_STACK_V1_BOOTSTRAP.csv
V4_STRUCTURED_FACTOR_STACK_V1_STAGE2_COEFFICIENTS.csv   # only if frozen
V4_STRUCTURED_FACTOR_STACK_V1_LINEAGE.json
V4_STRUCTURED_FACTOR_STACK_V1_BLOCK_SCHEMA.json
```

If status is `...FROZEN_READY_FOR_FRESH_CONFIRM_DEV`, the next step is to freeze a completely fresh subject roster before event collection. If status is `...REJECTED_NO_FRESH_CONFIRM`, stop and do not collect a new corpus for this architecture.